# Medical Prediction Model Evaluation Toolkit

A comprehensive walkthrough of the evaluation framework from **Van Calster et al. (2025)**, *"Evaluation of performance measures in predictive AI models to support medical decisions"*, The Lancet Digital Health.

This notebook demonstrates all 32 performance measures across 5 domains, 7 visualization types, logistic recalibration, and bootstrap confidence intervals — using the ADNEX ovarian tumour prediction model as a case study.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # ensure plots work headless; notebook will still render inline
import matplotlib.pyplot as plt

# Core evaluation
from medpred.evaluate import evaluate_model, evaluate_with_ci, results_to_dataframe

# Individual metric modules
from medpred.metrics.discrimination import discrimination_metrics, roc_curve_data, pr_curve_data
from medpred.metrics.calibration import calibration_metrics, logistic_recalibration, calibration_curve_data
from medpred.metrics.overall import overall_metrics
from medpred.metrics.classification import classification_metrics
from medpred.metrics.clinical_utility import clinical_utility_metrics, net_benefit_curve, expected_cost_curve

# Visualization
from medpred.visualization.plots import (
    plot_roc_curve, plot_pr_curve, plot_calibration,
    plot_risk_distribution, plot_decision_curve,
    plot_expected_cost_curve, plot_classification_at_thresholds,
    plot_full_evaluation,
)

# Data loading & bootstrap
from medpred.utils.data import load_case_study_data
from medpred.utils.bootstrap import bootstrap_ci

## 1. Load and explore the data

The ADNEX case study: external validation of the ADNEX model for predicting malignancy in women with an ovarian tumour (894 patients).

In [ ]:
y_true, y_prob = load_case_study_data()

print(f"Patients:     {len(y_true)}")
print(f"Malignant:    {np.sum(y_true)} ({100*np.mean(y_true):.1f}%)")
print(f"Benign:       {np.sum(y_true == 0)} ({100*np.mean(y_true == 0):.1f}%)")
print(f"\nPredicted probabilities — min: {y_prob.min():.4f}, median: {np.median(y_prob):.4f}, max: {y_prob.max():.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(y_prob, bins=50, edgecolor="white", alpha=0.8)
axes[0].set_xlabel("Predicted Probability")
axes[0].set_ylabel("Count")
axes[0].set_title("Distribution of Predicted Probabilities")

axes[1].hist(y_prob[y_true == 0], bins=50, alpha=0.6, label="Benign")
axes[1].hist(y_prob[y_true == 1], bins=50, alpha=0.6, label="Malignant")
axes[1].set_xlabel("Predicted Probability")
axes[1].set_ylabel("Count")
axes[1].set_title("Predictions by Outcome")
axes[1].legend()

plt.tight_layout()
plt.show()

## 2. Full model evaluation (one-liner)

`evaluate_model` computes all 32 metrics across the 5 performance domains in a single call.

**Decision threshold = 0.1** (10%) — accepting up to 9 false positives per true positive, appropriate for a cancer screening context.

In [ ]:
results = evaluate_model(y_true, y_prob, threshold=0.1)

# Quick look at the structure
for domain, metrics in results.items():
    if domain == "meta":
        continue
    print(f"\n{domain.upper()} ({len(metrics)} metrics)")
    for k, v in metrics.items():
        if isinstance(v, float):
            print(f"  {k:30s} {v:.4f}")
        else:
            print(f"  {k:30s} {v}")

### Results as a DataFrame

The `results_to_dataframe` function converts results into a table annotated with **properness** (can the measure be fooled by a non-ideal model?), **focus** (clear decision-analytical interpretation?), and the paper's **recommendation**.

In [ ]:
df = results_to_dataframe(results)
df["value"] = df["value"].round(4)
df

## 3. Domain-by-domain deep dive

Each domain can also be computed individually using the lower-level metric functions.

### 3.1 Discrimination

Can the model distinguish between events and non-events? The paper **recommends AUROC** and considers AUPRC and pAUROC inadvisable.

In [ ]:
disc = discrimination_metrics(y_true, y_prob, min_sensitivity=0.8)

for k, v in disc.items():
    tag = " ✓ Recommended" if k == "auroc" else " ✗ Inadvisable"
    print(f"  {k:10s} = {v:.4f}{tag}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
plot_roc_curve(y_true, y_prob, auroc=disc["auroc"], ax=axes[0])
plot_pr_curve(y_true, y_prob, auprc=disc["auprc"], ax=axes[1])
plt.tight_layout()
plt.show()

### 3.2 Calibration

Do predicted probabilities match observed event rates? The paper **recommends the calibration plot** (visual assessment), not any single number.

In [ ]:
cal = calibration_metrics(y_true, y_prob)

print("Calibration metrics (all semi-proper, clear focus):")
print(f"  O:E ratio             = {cal['oe_ratio']:.4f}  (ideal = 1.0)")
print(f"  Calibration intercept = {cal['calibration_intercept']:.4f}  (ideal = 0)")
print(f"  Calibration slope     = {cal['calibration_slope']:.4f}  (ideal = 1.0)")
print(f"  ECI                   = {cal['eci']:.4f}  (ideal = 0)")
print(f"  ICI                   = {cal['ici']:.4f}  (ideal = 0)")
print(f"  ECE                   = {cal['ece']:.4f}  (ideal = 0)")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
plot_calibration(y_true, y_prob, ax=ax)
plt.show()

### 3.3 Overall performance

Composite measures that combine discrimination and calibration. Strictly proper measures (Brier, logloss) cannot be "fooled" — they are minimized only by the true probabilities.

In [ ]:
overall = overall_metrics(y_true, y_prob)

print("Strictly proper:")
print(f"  Log-likelihood        = {overall['loglikelihood']:.4f}")
print(f"  Log loss              = {overall['logloss']:.4f}")
print(f"  Brier score           = {overall['brier_score']:.4f}")

print("\nAsymptotically strictly proper:")
print(f"  Scaled Brier          = {overall['scaled_brier']:.4f}")
print(f"  McFadden R²           = {overall['mcfadden_r2']:.4f}")
print(f"  Cox-Snell R²          = {overall['cox_snell_r2']:.4f}")
print(f"  Nagelkerke R²         = {overall['nagelkerke_r2']:.4f}")

print("\nImproper (inadvisable):")
print(f"  Discrimination slope  = {overall['discrimination_slope']:.4f}")
print(f"  MAPE                  = {overall['mape']:.4f}")

### 3.4 Classification

Threshold-based measures. The paper notes that **all classification summary measures are improper** at clinically relevant thresholds. F1 is the only measure that is both improper **and** has unclear focus — making it the least recommended.

Sensitivity/specificity and PPV/NPV are acceptable when reported **descriptively as pairs**.

In [ ]:
clf = classification_metrics(y_true, y_prob, threshold=0.1)

print(f"Threshold: {clf['threshold']}")
print(f"Confusion matrix: TP={clf['tp']}, FP={clf['fp']}, FN={clf['fn']}, TN={clf['tn']}")

print("\nDescriptive pairs:")
print(f"  Sensitivity = {clf['sensitivity']:.4f}   Specificity = {clf['specificity']:.4f}")
print(f"  PPV         = {clf['ppv']:.4f}   NPV         = {clf['npv']:.4f}")

print("\nSummary measures (all improper at clinical thresholds):")
for k in ["accuracy", "balanced_accuracy", "youden_index", "kappa", "mcc", "diagnostic_odds_ratio", "f1_score"]:
    print(f"  {k:25s} = {clf[k]:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
plot_classification_at_thresholds(y_true, y_prob, ax=ax)
ax.axvline(x=0.1, color="gray", linestyle=":", alpha=0.7, label="t = 0.1")
ax.legend()
plt.show()

### 3.5 Clinical utility

The **most important domain** per the paper. Net benefit and expected cost incorporate misclassification costs, answering: *"Is this model better than treating all or treating none?"*

In [ ]:
cu = clinical_utility_metrics(y_true, y_prob, threshold=0.1, cost_fn_ratio=0.9)

print("Recommended measures (all semi-proper, clear focus):")
print(f"  Net benefit (t=0.1)        = {cu['net_benefit']:.4f}  (max = prevalence = {cu['prevalence']:.4f})")
print(f"  Standardized NB (t=0.1)    = {cu['standardized_net_benefit']:.4f}  (max = 1.0)")
print(f"  Expected cost (cfn=0.9)    = {cu['expected_cost']:.4f}")
print(f"  Optimal EC threshold       = {cu['expected_cost_threshold']:.4f}")
print(f"\nReference strategies:")
print(f"  Treat-all NB (t=0.1)       = {cu['treat_all_nb']:.4f}")
print(f"  Treat-none NB              = {cu['treat_none_nb']:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_decision_curve(y_true, y_prob, ax=axes[0])
plot_expected_cost_curve(y_true, y_prob, ax=axes[1])
plt.tight_layout()
plt.show()

## 4. Risk distribution

The paper **recommends** reporting risk distributions by outcome. This violin plot shows how predicted probabilities separate events from non-events.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
plot_risk_distribution(y_true, y_prob, ax=ax)
plt.show()

## 5. Full evaluation panel

A single call generates all 6 recommended plots in a 2×3 grid.

In [ ]:
fig = plot_full_evaluation(y_true, y_prob, auroc=results["discrimination"]["auroc"])
plt.show()

## 6. Logistic recalibration (Platt scaling)

Recalibration adjusts predicted probabilities to better match observed rates. It is **rank-preserving**, so discrimination (AUROC) stays the same while calibration improves.

In [ ]:
y_prob_recal = logistic_recalibration(y_true, y_prob)
results_recal = evaluate_model(y_true, y_prob_recal, threshold=0.1)

# Compare key metrics before and after recalibration
comparison = pd.DataFrame({
    "Measure": ["AUROC", "O:E ratio", "Cal. intercept", "Cal. slope",
                "Brier score", "Net benefit (t=0.1)"],
    "Before": [
        results["discrimination"]["auroc"],
        results["calibration"]["oe_ratio"],
        results["calibration"]["calibration_intercept"],
        results["calibration"]["calibration_slope"],
        results["overall"]["brier_score"],
        results["clinical_utility"]["net_benefit"],
    ],
    "After": [
        results_recal["discrimination"]["auroc"],
        results_recal["calibration"]["oe_ratio"],
        results_recal["calibration"]["calibration_intercept"],
        results_recal["calibration"]["calibration_slope"],
        results_recal["overall"]["brier_score"],
        results_recal["clinical_utility"]["net_benefit"],
    ],
    "Ideal": ["higher", "1.0", "0", "1.0", "lower", "higher"],
})
comparison.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
plot_calibration(y_true, y_prob, ax=axes[0])
axes[0].set_title("Calibration — Before Recalibration")
plot_calibration(y_true, y_prob_recal, ax=axes[1])
axes[1].set_title("Calibration — After Recalibration")
plt.tight_layout()
plt.show()

## 7. Bootstrap confidence intervals

`evaluate_with_ci` wraps the full evaluation with percentile bootstrap confidence intervals.

In [ ]:
results_ci = evaluate_with_ci(
    y_true, y_prob,
    threshold=0.1,
    n_bootstrap=500,  # use 1000+ for publication
    ci_level=0.95,
    random_state=42,
)

# Display key results with CIs
key_metrics = [
    ("discrimination", "auroc", "AUROC"),
    ("calibration", "oe_ratio", "O:E Ratio"),
    ("calibration", "calibration_intercept", "Cal. Intercept"),
    ("calibration", "calibration_slope", "Cal. Slope"),
    ("overall", "brier_score", "Brier Score"),
    ("overall", "scaled_brier", "Scaled Brier"),
    ("clinical_utility", "net_benefit", "Net Benefit"),
    ("clinical_utility", "standardized_net_benefit", "Std. Net Benefit"),
]

ci_rows = []
for domain, metric, label in key_metrics:
    v = results_ci[domain][metric]
    ci_rows.append({
        "Measure": label,
        "Point Estimate": round(v["point"], 4),
        "95% CI Lower": round(v["lower"], 4),
        "95% CI Upper": round(v["upper"], 4),
    })

pd.DataFrame(ci_rows)

In [ ]:
# Full results table with CIs
df_ci = results_to_dataframe(results_ci, with_ci=True)
df_ci.head(15)

## 8. Lower-level API: bootstrap a single domain, access raw curve data

You can bootstrap any single metric module directly, and all curve functions return raw arrays for custom plotting.

In [ ]:
# Bootstrap just the discrimination metrics
disc_ci = bootstrap_ci(
    y_true, y_prob,
    metric_fn=discrimination_metrics,
    n_bootstrap=500,
    ci_level=0.95,
    random_state=42,
    min_sensitivity=0.8,
)

for metric, val in disc_ci.items():
    print(f"  {metric:10s}: {val['point']:.4f}  (95% CI: {val['lower']:.4f} – {val['upper']:.4f})")

In [ ]:
# ROC and PR curve data
roc = roc_curve_data(y_true, y_prob)
pr = pr_curve_data(y_true, y_prob)

print(f"ROC curve: {len(roc['fpr'])} points")
print(f"PR curve:  {len(pr['precision'])} points")

# Calibration curve data
cal_curve = calibration_curve_data(y_true, y_prob, n_groups=10)
print(f"\nGrouped calibration ({len(cal_curve['grouped_pred'])} groups):")
for p, o in zip(cal_curve["grouped_pred"], cal_curve["grouped_obs"]):
    print(f"  predicted={p:.3f}  observed={o:.3f}")

In [ ]:
# Net benefit and expected cost curve data
nb = net_benefit_curve(y_true, y_prob)
ec = expected_cost_curve(y_true, y_prob)

print(f"Net benefit curve: {len(nb['thresholds'])} thresholds")
print(f"Expected cost curve: {len(ec['cost_ratios'])} cost ratios")

## 9. Paper recommendations summary

| What to report | Why |
|---|---|
| **AUROC** | Best single discrimination measure (semi-proper, clear focus) |
| **Calibration plot** | Visual assessment captures nuances no single number can |
| **Net benefit / decision curve** | Shows clinical utility across thresholds |
| **Risk distributions** | Reveals how the model behaves for events vs non-events |

| What to avoid | Why |
|---|---|
| **F1 score** | Only measure that is both *improper* and has *unclear focus* |
| **AUPRC** | Unclear decision-analytical focus |
| **Accuracy** at clinical thresholds | Improper — can be optimized by a non-ideal model |